# Preprocess

In [14]:
import re
import nltk
from nltk.stem import WordNetLemmatizer

class TextPreProc():
    def __init__(self, stopwords_path = "./english-stopwords.txt" , root_finding_method = "lemmatization"):
        try:
            with open(stopwords_path, 'r', encoding='utf-8') as f:
                self.stopwords = set(f.read().splitlines())
        except FileNotFoundError:
            self.stopwords = set()
        self.lemmatizer = WordNetLemmatizer()
        self.stemmer = nltk.PorterStemmer()
        self.root_finding_method = root_finding_method
        
    
    def remove_links_and_tags(self , text):
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'#\w+', '', text)
        text = re.sub(r'RT', '', text)
        text = re.sub(r'[\U00010000-\U0010ffff]', '', text)
        return text

    def remove_noise_characters(self , text):
        if not isinstance(text, str): return ""
        text = re.sub(r'<.*?>', ' ', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def clean_words(self, text): 
        words = text.split()
        processed_words = []
        
        for word in words:
            if word in self.stopwords:
                continue
            
            if self.root_finding_method == "lemmatization":
                final_word = self.lemmatizer.lemmatize(word)
            elif self.root_finding_method == "stemming":
                final_word = self.stemmer.stem(word)
            else:
                final_word = word
            processed_words.append(final_word)
        return ' '.join(processed_words)

    def process_text(self , text):
        text = text.lower()
        text = self.remove_links_and_tags(text)
        text = self.remove_noise_characters(text)
        text = self.clean_words(text)
        return text

# Data vectorization

In [15]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

class Vectorization:
    def __init__(self , dataset_path = "amazon_reviews.csv" , root_finding_method = "lemmatization" , model_path = "./local_model"):
        self.model = SentenceTransformer(model_path)
        self.dataset_path = dataset_path
        self.df = pd.read_csv(self.dataset_path)
        self.raw_col_name = self.df.columns[0]
        self.preprocessor = TextPreProc(root_finding_method = root_finding_method)

    def extract_fields(self, row):
        row_str = str(row)
        summary_match = re.search(r'Summary:\s*(.*?)\n', row_str)
        summary = summary_match.group(1).strip() if summary_match else ""
        text_match = re.search(r'Text:\s*(.*)', row_str, re.DOTALL)
        text = text_match.group(1).strip() if text_match else ""
        return pd.Series([summary, text], index=['Summary', 'Text'])
    
    def vectorize_texts(self):
        extracted = self.df[self.raw_col_name].apply(self.extract_fields)
        # extract the summary and text
        self.df['Summary'] = extracted['Summary']
        self.df['Text'] = extracted['Text']

        # clean the summary and text
        self.df['Cleaned_Summary'] = self.df['Summary'].apply(self.preprocessor.process_text)
        self.df['Cleaned_Text'] = self.df['Text'].apply(self.preprocessor.process_text)
        
        # combine the summary and text
        self.df['Final_Content'] = self.df['Cleaned_Summary'] + ". " + self.df['Cleaned_Text']
        embeddings = self.model.encode(self.df['Final_Content'].tolist(), show_progress_bar=True)
        self.df.to_csv("./amazon_reviews_cleaned.csv" , index=False)
        return self.df, embeddings

In [17]:
# df, X = Vectorization(root_finding_method = "lemmatization").vectorize_texts()

df, X = Vectorization(root_finding_method = "stemming").vectorize_texts()


print(X.shape)
print(df.shape)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Batches: 100%|██████████| 86/86 [00:52<00:00,  1.65it/s]


(2732, 384)
(2732, 6)
